In [10]:
!pip install python-docx -q
!pip install openpyxl -q

In [11]:
import os
import re
import pandas as pd
from docx import Document
import json

def clean_text(text):
    # Khử các ký tự đặc biệt gây lỗi Regex và chuẩn hóa khoảng trắng
    text = re.sub(r'\\', '', text)
    return text.strip()

def get_next_non_empty_line(lines, current_index):
    """Tìm dòng có nội dung tiếp theo để tránh lỗi dòng trống giữa tiêu đề"""
    for i in range(current_index + 1, len(lines)):
        if lines[i].strip():
            return lines[i].strip(), i
    return "", current_index

def parse_legal_docx(folder_path, excel_metadata_path):
    # 1. Đọc Metadata từ Excel
    df_meta = pd.read_excel(excel_metadata_path)
    # Chuyển thành dictionary để tra cứu nhanh theo ID_law
    meta_dict = df_meta.set_index('ID_law').to_dict('index')

    final_results = []

    # 2. Duyệt qua các file trong folder
    for filename in os.listdir(folder_path):
        if filename.endswith(".docx"):
            id_law = filename.replace(".docx", "")
            file_path = os.path.join(folder_path, filename)

            # Lấy thông tin meta từ Excel nếu tồn tại
            file_meta = meta_dict.get(id_law, {})

            doc = Document(file_path)
            # Chuyển nội dung docx thành list các dòng và làm sạch
            lines = [line.text.strip() for line in doc.paragraphs if line.text.strip()]

            current_phan = ""
            current_chuong = ""
            current_muc = ""
            current_tieu_muc = ""

            i = 0
            while i < len(lines):
                line = lines[i]

                # --- NHẬN DIỆN CẤU TRÚC PHÂN CẤP ---

                # PHẦN: Nhận diện "Phần thứ..."
                phan_match = re.match(r'^Phần thứ\s+([a-zA-Zà-ỹ\s]+)$', line, re.IGNORECASE)
                if phan_match:
                    next_val, next_idx = get_next_non_empty_line(lines, i)
                    current_phan = f"{line}: {next_val}"
                    current_chuong = current_muc = current_tieu_muc = ""
                    i = next_idx + 1
                    continue

                # CHƯƠNG: Nhận diện "Chương [Số La Mã]"
                chuong_match = re.match(r'^Chương\s+([IVXLCDM]+)$', line, re.IGNORECASE)
                if chuong_match:
                    next_val, next_idx = get_next_non_empty_line(lines, i)
                    current_chuong = f"{line}: {next_val}"
                    current_muc = current_tieu_muc = ""
                    i = next_idx + 1
                    continue

                # MỤC
                muc_match = re.match(r'^Mục\s+(\d+)\.?\s*(.*)$', line, re.IGNORECASE)
                if muc_match:
                    num, title = muc_match.groups()
                    current_muc = f"Mục {num}: {title.strip()}" if title else f"Mục {num}"
                    current_tieu_muc = ""
                    i += 1
                    continue

                # TIỂU MỤC
                tieu_muc_match = re.match(r'^Tiểu mục\s+(\d+)\.?\s*(.*)$', line, re.IGNORECASE)
                if tieu_muc_match:
                    num, title = tieu_muc_match.groups()
                    current_tieu_muc = f"Tiểu mục {num}: {title.strip()}" if title else f"Tiểu mục {num}"
                    i += 1
                    continue

                # --- TÁCH ĐIỀU VÀ NỘI DUNG ---
                # Gia cố Regex Điều: Tách riêng "Điều X" và "Nội dung tiêu đề"
                dieu_match = re.match(r'^(Điều\s+(\d+))\.\s*(.*)$', line, re.IGNORECASE)
                if dieu_match:
                    prefix, stt_dieu, title_dieu = dieu_match.groups()
                    full_article_title = f"{prefix}. {title_dieu}".strip()

                    id_article = f"{id_law}_D{stt_dieu}"
                    noi_dung_list = []

                    # Nếu tiêu đề điều có nội dung đi kèm, có thể xử lý tùy ý,
                    # ở đây ta vẫn giữ nguyên style cũ là gộp vào Article

                    j = i + 1
                    while j < len(lines):
                        # Điểm dừng: Gặp tiêu đề phân cấp khác
                        if (re.match(r'^(Điều|Chương|Mục|Tiểu mục|Phần thứ)\s+', lines[j], re.IGNORECASE)):
                            break
                        noi_dung_list.append(lines[j])
                        j += 1

                    # Tổng hợp dữ liệu kèm Metadata
                    item = {
                        "ID_law": id_law,
                        "ID_article": id_article,
                        "Part": current_phan,
                        "Chapter": current_chuong,
                        "Section": current_muc,
                        "MiniSection": current_tieu_muc,
                        "Article": full_article_title,
                        "Content": "\n".join(noi_dung_list).strip()
                    }

                    # Merge thông tin từ Excel vào JSON
                    item.update(file_meta)

                    final_results.append(item)
                    i = j
                    continue

                i += 1

    return final_results



In [12]:
folder_docx = "/content/drive/MyDrive/SE365/Final Project/Legal Data-docx"
excel_path = "/content/drive/MyDrive/SE365/Final Project/SE365/MetaData.xlsx"


In [13]:
df_meta = pd.read_excel(excel_path)
df_meta.head()
for col in df_meta.columns:
    dup_count = df_meta[col].duplicated().sum()

    if dup_count > 0:
        print(f"{col}: {dup_count} giá trị trùng")

LoaiVanBan: 67 giá trị trùng
NoiBanHanh: 66 giá trị trùng
NgayBanHanh: 16 giá trị trùng


In [14]:
df_meta[df_meta["ID_law"].duplicated(keep=False)]

,SoHieu,ID_law,LoaiVanBan,LinhVucNganh,NoiBanHanh,NgayBanHanh,ChuDe


In [15]:
data = parse_legal_docx(folder_docx, excel_path)
with open('/content/drive/MyDrive/SE365/Final Project/Newdata-27-5-2025/extract_step1.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

In [16]:
print(f"📊 Tổng số lượng Điều luật đã tách được: {len(data)}")
from collections import Counter
file_counts = Counter(item['ID_law'] for item in data)
print("\n📑 Số lượng Điều theo từng văn bản:")
for law_id, count in file_counts.items():
    print(f" - {law_id}: {count} điều")

📊 Tổng số lượng Điều luật đã tách được: 8089

📑 Số lượng Điều theo từng văn bản:
 - 135_VBHN-VPQH: 407 điều
 - 103_2016_QH13: 61 điều
 - 15_2020_ND-CP: 124 điều
 - 24_2023_QH15: 73 điều
 - 91_2015_QH13: 689 điều
 - 101_2015_QH13: 498 điều
 - 92_2015_QH13: 517 điều
 - 67_2006_QH11: 71 điều
 - 86_2015_QH13: 54 điều
 - 20_2023_QH15: 53 điều
 - 116_2025_QH15: 45 điều
 - 88_VBHN-VPQH: 43 điều
 - 01_2021_ND-CP: 101 điều
 - 01_2026_QH16: 31 điều
 - 03_2026_QH16: 30 điều
 - 08_2020_ND-CP: 75 điều
 - 08_2022_QH15: 157 điều
 - 09_2017_QH14: 78 điều
 - 11_2022_QH15: 118 điều
 - 15_2012_QH13: 142 điều
 - 16_2023_QH15: 75 điều
 - 36_2005_QH11: 324 điều
 - 41_2019_QH14: 207 điều
 - 45_2013_QH13: 212 điều
 - 46_2017_ND-CP: 113 điều
 - 48_2019_QH14: 50 điều
 - 45_2019_QH14: 220 điều
 - 50_2014_QH13: 168 điều
 - 51_2014_QH13: 133 điều
 - 52_2010_QH12: 52 điều
 - 53_2014_QH13: 81 điều
 - 54_2019_QH14: 135 điều
 - 55_VBHN-VPQH: 89 điều
 - 57_2014_QH13: 102 điều
 - 58_2014_QH13: 125 điều
 - 59_2020_QH14: 

In [17]:
import json

# 1. Đọc dữ liệu từ file json cũ
with open('/content/drive/MyDrive/SE365/Final Project/Newdata-27-5-2025/extract_step1.json', 'r', encoding='utf-8') as f:
    old_data = json.load(f)

new_data = []
for item in old_data:
    new_item = {}
    new_item['id'] = item.get('ID_article')
    for key, value in item.items():
        if key not in ['ID_law', 'ID_article']:
            new_item[key] = value

    new_data.append(new_item)
# 3. Lưu lại vào file mới hoặc ghi đè lên file cũ
with open('/content/drive/MyDrive/SE365/Final Project/Newdata-27-5-2025/change_step2.json', 'w', encoding='utf-8') as f:
    json.dump(new_data, f, ensure_ascii=False, indent=4)

print(f"✅ Đã xử lý xong!")
print(f"📊 Tổng số lượng bản ghi (id): {len(new_data)}")
print(f"📂 File đã được lưu tại: data_luat_final_v2.json")

# Kiểm tra thử cấu trúc của bản ghi đầu tiên
if new_data:
    print("\n📝 Mẫu bản ghi sau khi đổi:")
    print(json.dumps(new_data[0], ensure_ascii=False, indent=2))

✅ Đã xử lý xong!
📊 Tổng số lượng bản ghi (id): 8089
📂 File đã được lưu tại: data_luat_final_v2.json

📝 Mẫu bản ghi sau khi đổi:
{
  "id": "135_VBHN-VPQH_D1",
  "Part": "Phần thứ nhất: NHỮNG QUY ĐỊNH CHUNG",
  "Chapter": "Chương I: ĐIỀU KHOẢN CƠ BẢN",
  "Section": "",
  "MiniSection": "",
  "Article": "Điều 1. Nhiệm vụ của Bộ luật Hình sự",
  "Content": "Bộ luật Hình sự có nhiệm vụ bảo vệ chủ quyền quốc gia, an ninh của đất nước, bảo vệ chế độ xã hội chủ nghĩa, quyền con người, quyền công dân, bảo vệ quyền bình đẳng giữa đồng bào các dân tộc, bảo vệ lợi ích của Nhà nước, tổ chức, bảo vệ trật tự pháp luật, chống mọi hành vi phạm tội; giáo dục mọi người ý thức tuân theo pháp luật, phòng ngừa và đấu tranh chống tội phạm.\nBộ luật này quy định về tội phạm và hình phạt.",
  "SoHieu": "135/VBHN-VPQH",
  "LoaiVanBan": "Bộ Luật",
  "LinhVucNganh": "Hình Sự",
  "NoiBanHanh": "Văn Phòng Quốc Hội",
  "NgayBanHanh": "05/09/2025",
  "ChuDe": "Bộ Luật Hình Sự"
}


In [18]:
import json

desired_order = [
    "id", "Part", "Chapter", "Section", "MiniSection",
    "Article", "Content", "LinhVucNganh", "LoaiVanBan",
    "NoiBanHanh", "SoHieu", "NgayBanHanh", "ChuDe"
]

def reorder_json_v2(input_path, output_path):
    # 2. Đọc file v2 vào
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    reordered_data = []

    for item in data:
        # Tạo dictionary mới để giữ thứ tự chèn
        clean_item = {}

        # Duyệt qua danh sách thứ tự mong muốn
        for key in desired_order:
            # Lấy giá trị từ item, nếu không tồn tại thì gán chuỗi rỗng
            clean_item[key] = item.get(key, "")

        reordered_data.append(clean_item)

    # 3. Lưu lại file đã sắp xếp thứ tự
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(reordered_data, f, ensure_ascii=False, indent=4)

    print(f"✅ Đã sắp xếp lại thứ tự thuộc tính cho {len(reordered_data)} bản ghi.")

# Thực thi
reorder_json_v2('/content/drive/MyDrive/SE365/Final Project/Newdata-27-5-2025/change_step2.json', '/content/drive/MyDrive/SE365/Final Project/Newdata-27-5-2025/final_data.json')

✅ Đã sắp xếp lại thứ tự thuộc tính cho 8089 bản ghi.
